# Python For ML Session 5: Machine Learning Workflow

Last sessions involved desribing the dataset with help of pandas. In the last slides, we also discussed about **predicting** from data. Concepts of features and labels, train/test data split, training, loss, and difference between classification and regression.

This session we turn those ideas into **code**. We will reuse the penguins dataset you already know and walk the full machine-learning workflow end to end. This session we will just focus on the workflow and not the maths and theory behind the algorithms. By the end of this session, you should be able to take a table of data and get a working prediction out of it.

## The ML workflow at a glance

Almost every supervised ML task follows the same six steps. We will do each one in order:

1. **Load & clean** the data &nbsp;→&nbsp; `read_csv`, `dropna`
2. **Choose features (X) and a label (y)** &nbsp;→&nbsp; what we know vs. what we predict
3. **Split** into train and test &nbsp;→&nbsp; `train_test_split` method  of scikit learn library, which splits the dataset into training and testing sets.
4. **Choose a model and train it** &nbsp;→&nbsp; `model.fit(...)` using scikit learn library.
5. **Predict** on data the model never saw &nbsp;→&nbsp; `model.predict(...)`
6. **Evaluate** the result &nbsp;→&nbsp; `model.score(...)`

All the cells below cover these steps.

## 1. Setting up the libraries and the dataset

We import the tools we need and load the penguins dataset, exactly like in Session 4. Machine-learning models cannot handle missing values, so we drop the rows with `NaN` first.

In [ ]:
# Importing pandas
import pandas as pd

# Load the penguins dataset (same as Session 4) and drop rows with missing values
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
df = df.dropna()

df.shape

(333, 7)

In [25]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


## 2. Features (X) and Label (y)

From the slides:

- **Features (X)** = The columns or values that the model uses to generate prediction
- **Label (y)** = Represents the column which we want to be predicted.

We'll start with a **classification** task: predict a penguin's **species** from its four numeric measurements. So `X` is the four measurement columns, and `y` is the `species` column.

<br>

![Inter Quartile Range](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/FeatureLabel.png)
_fig: Feature and Label splitting from the dataset_

```python
X = df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
y = df["species"]
```

`X` is a DataFrame (many columns), `y` is a single Series.

Notice this is just column selection from pandas.

In [26]:
# Features: the four numeric measurements (what we know)
X = df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]

# Label: the species (what we want to predict)
y = df["species"]

print("X shape (rows, features):", X.shape)
print("y shape (rows):", y.shape)
print("Classes we can predict:", y.unique())

X shape (rows, features): (333, 4)
y shape (rows): (333,)
Classes we can predict: ['Adelie' 'Chinstrap' 'Gentoo']


## 3. Train / test split

![](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/TrainTestSplit.png)
_fig: Splitting testing and training dataset, splitting features and labels for both training and testing dataset_

From the slides: we hold back some data the model never sees, then grade it on that. This is how we check the model can **generalize** instead of just memorizing.

`train_test_split` shuffles the rows and cuts them into a training portion and a test portion. `test_size=0.2` means 20% is held back for testing. `random_state=42` just makes the random shuffle repeatable, so everyone gets the same split.

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

In [27]:
# sklearn library allows us to do train test split easily
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Test rows:    ", X_test.shape[0])

print(type(y_train))


Training rows: 266
Test rows:     67
<class 'pandas.core.series.Series'>


## 4. Choose a model and train it

Remember the "who writes the rules?" slide — the hand-written `if flipper > 215 and bill < 45: ...`? A **decision tree** learns those yes/no rules from the data *for you*. Decision tree therefore is the most intuitive model concept wise following that.

Two lines do the whole thing:

```python
model = DecisionTreeClassifier(max_depth=3)   # create the model (the knobs are empty)
model.fit(X_train, y_train)                    # TRAINING: tune the knobs to fit the data
```

`max_depth=3` is a **hyperparameter** — a setting *we* choose (how many questions deep the tree can go). `.fit(...)` is the training step from the slides: this is where the model adjusts its internal rules to match the examples.

<br>

![Decision tree 3 depth](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/DecisionTree.png)

_fig: Decision Tree training_

In [29]:
# We are importing the Model from sklearn library too
from sklearn.tree import DecisionTreeClassifier

# Create the model
model = DecisionTreeClassifier(max_depth=3, random_state=0)

# Train it on the TRAINING data only
model.fit(X_train, y_train)

print("Model trained!")


Model trained!


## 5. Predict

A trained model can now make predictions on data it has never seen. This step is called **inference**.

```python
predictions = model.predict(X_test)
```

We can also feed it a brand-new penguin we invent, as long as we give it the same four columns.

In [30]:
# Predict the species for every penguin in the test set
predictions = model.predict(X_test)

# Compare the first 10 predictions with the true answers
comparison = pd.DataFrame({
    "predicted": predictions[:10],
    "actual":    y_test.values[:10]
})
print(comparison)

# Predict a single brand-new penguin
new_penguin = pd.DataFrame([{
    "bill_length_mm": 45.0,
    "bill_depth_mm": 17.0,
    "flipper_length_mm": 210.0,
    "body_mass_g": 4200.0
}])
print("New penguin is predicted to be:", model.predict(new_penguin)[0])

   predicted     actual
0     Adelie     Adelie
1     Gentoo     Gentoo
2     Adelie     Adelie
3  Chinstrap  Chinstrap
4     Adelie     Adelie
5     Gentoo     Gentoo
6     Gentoo     Gentoo
7  Chinstrap  Chinstrap
8  Chinstrap  Chinstrap
9  Chinstrap  Chinstrap
New penguin is predicted to be: Gentoo


## 6. Evaluate: obtain the result

![Calculating metrics](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/Metrics.png)

*Fig - Calculating the metrics*

<br>

From the slides: we grade the model on the test set it never saw. For classification, the simplest result is **accuracy**, the fraction of predictions it got right to total predictions.

There are two equivalent ways to get it:

```python
accuracy_score(y_test, predictions)   # compare predictions to the truth
model.score(X_test, y_test)           # shortcut: predict + score in one call
```

A **confusion matrix** shows breakdown of prediction based on classes, where the model misclassified a penguin and what it misclassified the penguin species as.


In [31]:
# These are the metrics that we use to measure the performance of our models
from sklearn.metrics import accuracy_score, confusion_matrix, r2_score, mean_squared_error

# Accuracy two ways (they give the same number)
print("Accuracy (accuracy_score):", round(accuracy_score(y_test, predictions), 3))
print("Accuracy (model.score):   ", round(model.score(X_test, y_test), 3))

# Confusion matrix: rows = true species, columns = predicted species
print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions, labels=model.classes_))
print("Order of labels:", list(model.classes_))

Accuracy (accuracy_score): 0.985
Accuracy (model.score):    0.985

Confusion matrix:
[[31  0  0]
 [ 1 12  0]
 [ 0  0 23]]
Order of labels: ['Adelie', 'Chinstrap', 'Gentoo']


### Task 1

Using the steps above, experiment with the model and watch the accuracy change:

- Change `max_depth` to `1`, then to `10`. Retrain and re-score each time. What happens to accuracy? (A depth of 1 is *too simple*, the model underfits, that is the model is too simple to capture the underlying patterns of the data.)
- Try training with only **two** features instead of four (e.g. just `flipper_length_mm` and `body_mass_g`). Does accuracy go up or down?

In [ ]:
# Task 1 — your experiments here
# Hint: rebuild the model with a different max_depth, call .fit() again, then .score()

model_2 = DecisionTreeClassifier(max_depth=5, random_state=0)

# Train it on the TRAINING data only
model_2.fit(X_train, y_train)

print("Accuracy (model.score):   ", round(model_2.score(X_test, y_test), 3))

Accuracy (model.score):    1.0


## Swapping the model

The best part of this workflow: **steps 1–5 don't change when you change the model.** Only one line is different. Let's try the model from the slide demo, `KNeighborsClassifier` (it predicts a penguin's species by looking at the most similar penguins it has already seen).

```python
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
model.score(X_test, y_test)
```

Run it and compare the accuracy to the decision tree.

In [33]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print("KNN accuracy:", round(knn.score(X_test, y_test), 3))

KNN accuracy: 0.821


You'll probably notice the KNN score is **lower** than the decision tree. That's not because KNN is a worse model. It's because KNN measures *distances* between penguins (Eucledean distance that you learnt in Optional Maths co-ordinates), and our features are on wildly different scales (body mass is in the thousands, bill length is in the tens). The big numbers drown out the small ones.

The fix is **feature scaling**, which is covered in the optional section at the end. This is a good example of why the *workflow* and the *data preparation* matter as much as the model choice.

### Task 2

`n_neighbors` (often called *k*) is a hyperparameter. Try a different values 5, 10, 15 and print the accuracy for each. Higher hyperparameter value doesn't mean that the result will be good?


In [ ]:
# Task 2 — try different values of n_neighbors
# Try a different values 5, 10, 15 and print the accuracy for each. Higher hyperparameter value doesn't mean that the result will be good?
for n in [5, 10, 15]:
    knn = KNeighborsClassifier(n_neighbors=n)
    knn.fit(X_train, y_train)
    print(f"KNN accuracy (n_neighbors={n}):", round(knn.score(X_test, y_test), 3))


KNN accuracy (n_neighbors=5): 0.821
KNN accuracy (n_neighbors=10): 0.806
KNN accuracy (n_neighbors=15): 0.821


## A regression example

Everything so far predicted a **category** (species) — that's classification. The exact same workflow predicts a **number** if we just change the label. Let's predict a penguin's **body mass** from its other measurements. That's **regression**, and the model is `LinearRegression`. This is similar to straight line y = mx + c, but w replaces m and b replaces c. In ML, it is commonly referred to as y = wx + b. w is called weights, and b is called bias, they are both parameters that the model learns with data.

The only differences are:
- `y` is now a numeric column (`body_mass_g`).
- We grade it with a regression metric instead of accuracy: **R²** (1.0 is perfect, 0 means "no better than guessing the average") and **RMSE** (the typical error, in grams).


In [36]:
# Features and a NUMERIC label this time, we are only using 2 numeric fields
X_reg = df[["bill_length_mm", "flipper_length_mm"]]
y_reg = df["body_mass_g"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Same workflow: create -> fit -> predict -> evaluate
from sklearn.linear_model import LinearRegression
reg = LinearRegression()
reg.fit(Xr_train, yr_train)
reg_predictions = reg.predict(Xr_test)

print("R^2 :", round(r2_score(yr_test, reg_predictions), 3))
print("RMSE:", round(mean_squared_error(yr_test, reg_predictions) ** 0.5, 1), "grams")

R^2 : 0.795
RMSE: 359.6 grams


### Task 3

- Add `bill_depth_mm` back into the features (so you use all available measurements). Does R² improve?
- Print the model's learned coefficients with `reg.coef_` and the intercept with `reg.intercept_`. These are the **parameters** (the `w` and `b`) the training found.


In [37]:
# Task 3 — your regression experiments here
# Add `bill_depth_mm` back into the features (so you use all available measurements). Does R² improve?
# Print the model's learned coefficients with `reg.coef_` and the intercept with `reg.intercept_`. These are the **parameters** (the `w` and `b`) the training found.

from sklearn.linear_model import LinearRegression
reg = LinearRegression()
reg.fit(Xr_train, yr_train)
print("Coefficients:", reg.coef_)
print("Intercept:", reg.intercept_)



Coefficients: [ 4.9958433  49.11369677]
Intercept: -5877.429125505831


## Recap: the workflow, and the words in code

You just ran the entire supervised ML workflow twice (once for classification, once for regression) and swapped models without changing the surrounding steps. Here is how the slide vocabulary maps to the code:

| Slide word | In code |
|---|---|
| Features (X) | the columns you select into `X` |
| Label (y) | the column you put in `y` |
| Train / test | `train_test_split(...)` |
| Model | `DecisionTreeClassifier()`, `KNeighborsClassifier()`, `LinearRegression()` |
| Training | `model.fit(X_train, y_train)` |
| Prediction / inference | `model.predict(X_new)` |
| Result / score | `model.score(...)`, `accuracy_score(...)`, `r2_score(...)` |
| Classification | predicting a category (species) |
| Regression | predicting a number (body mass) |

The headline: **the workflow is the same no matter the model or the task.** Learn it once, reuse it forever.

## More on this

**If you've finished the tasks above, read through these and try them in a new cell.**

### Feature scaling (why KNN scored low)

KNN compares penguins by distance, so features must be on a comparable scale. `StandardScaler` rescales every feature to have a similar spread. We chain it in front of the model with a `Pipeline` so the scaling is learned from the training data only (this avoids "data leakage").

```python
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

scaled_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
scaled_knn.fit(X_train, y_train)
scaled_knn.score(X_test, y_test)   # jumps dramatically once features are on equal footing
```

The same KNN that scored poorly unscaled climbs to good accuracy score once the features are scaled. The model didn't change; the *data preparation* did.

### Categorical features

We only used numeric columns. To also use `island` or `sex` (which are text), you'd convert them to numbers first — e.g. `pd.get_dummies(df, columns=["island", "sex"])`. This will convert the categorical columns to multiple columns with numeric values that the models can use. Most models can't read raw text.

### The split is random

Change `random_state` to a different number and re-run — the accuracy will wobble a little, because a different set of penguins lands in the test set. To get a more stable estimate, your average over several splits with **cross-validation**:

```python
from sklearn.model_selection import cross_val_score
cross_val_score(DecisionTreeClassifier(max_depth=3), X, y, cv=5)
```

### Predicting probabilities

Classifiers can tell you *how confident* they are, not just the final label:

```python
model.predict_proba(new_penguin)   # probability for each species
```


# Well Done!